# 线性神经网络总结

本notebook整理了《动手学深度学习》第三章的核心概念和代码，涵盖线性回归和Softmax回归。

## 本章结构
1. **线性回归** — 理论基础、损失函数、优化算法
2. **线性回归实现** — 从零实现 vs 简洁实现
3. **Softmax回归** — 分类问题、softmax运算、交叉熵损失
4. **图像分类数据集** — Fashion-MNIST

## 一、线性回归（Linear Regression）

线性回归是最简单且最流行的回归工具，用于预测**连续数值**（如房价、销量）。

### 1.1 线性模型

**核心公式**：给定$d$个特征，预测值为：

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b = \mathbf{w}^\top \mathbf{x} + b$$

- **权重（weight）** $\mathbf{w}$：每个特征对预测的影响程度
- **偏置（bias）** $b$：所有特征为0时的预测值（截距）
- **仿射变换（affine transformation）**：线性变换 + 平移

**批量表示**：$n$个样本，$d$个特征

$$\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + b, \quad \mathbf{X} \in \mathbb{R}^{n \times d}$$

### 1.2 损失函数（Loss Function）

**均方误差（Mean Squared Error, MSE）**：

$$L(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} \frac{1}{2}(\hat{y}^{(i)} - y^{(i)})^2$$

- 常数$\frac{1}{2}$是为了求导后系数为1
- 目标：找到$\mathbf{w}^*, b^* = \arg\min_{\mathbf{w}, b} L(\mathbf{w}, b)$

### 1.3 解析解（Analytical Solution）

线性回归存在闭式解：

$$\mathbf{w}^* = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**优点**：直接计算，无需迭代
**局限**：仅适用于简单问题，深度学习中通常不存在解析解

### 1.4 随机梯度下降（Stochastic Gradient Descent, SGD）

当不存在解析解时，使用**梯度下降**迭代更新参数：

$$(\mathbf{w}, b) \leftarrow (\mathbf{w}, b) - \frac{\eta}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \partial_{(\mathbf{w},b)} l^{(i)}(\mathbf{w}, b)$$

- **学习率（learning rate）** $\eta$：控制更新步长
- **批量大小（batch size）** $|\mathcal{B}|$：每次更新使用的样本数
- **超参数（hyperparameter）**：需要手动设定，不在训练中更新

### 1.5 正态分布与平方损失

**为什么用平方误差？** 假设噪声服从正态分布：

$$y = \mathbf{w}^\top \mathbf{x} + b + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

通过**极大似然估计（Maximum Likelihood Estimation, MLE）**，最小化均方误差等价于最大化似然函数。

### 1.6 神经网络表示

线性回归可视为**单层神经网络**：
- 输入层：$d$个特征
- 输出层：1个神经元
- **全连接层（fully-connected layer）**：每个输入都与输出相连

## 二、线性回归实现

### 2.1 从零实现（Scratch Implementation）

训练过程的核心要素：
1. **数据流水线**：生成/读取数据、小批量采样
2. **模型**：定义前向传播
3. **损失函数**：均方误差
4. **优化算法**：小批量SGD

In [ ]:
import torch
import random

# 1. 生成数据集
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
print(f"features shape: {features.shape}, labels shape: {labels.shape}")

In [ ]:
# 2. 读取数据集（小批量采样）
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)  # 随机打乱
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i:min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

# 3. 初始化模型参数
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 4. 定义模型
def linreg(X, w, b):
    """线性回归模型: y = Xw + b"""
    return torch.matmul(X, w) + b

# 5. 定义损失函数
def squared_loss(y_hat, y):
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

# 6. 定义优化算法
def sgd(params, lr, batch_size):
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()  # 清除梯度

In [ ]:
# 7. 训练
lr = 0.03
num_epochs = 3
batch_size = 10
net = linreg
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)  # 小批量损失
        l.sum().backward()          # 反向传播
        sgd([w, b], lr, batch_size) # 更新参数
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

print(f'\nw的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')

### 2.2 简洁实现（Concise Implementation）

使用PyTorch的高级API（`nn`模块）可以更简洁地实现：

```python
from torch import nn

# 定义模型
net = nn.Sequential(nn.Linear(2, 1))

# 初始化参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

# 定义损失函数
loss = nn.MSELoss()

# 定义优化器
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# 训练
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
```

## 三、Softmax回归（Softmax Regression）

Softmax回归用于**分类问题**，预测样本属于哪个类别（如图像分类）。

### 3.1 分类问题

**标签表示**：使用**独热编码（one-hot encoding）**
- 类别"猫"→ $(1, 0, 0)$
- 类别"鸡"→ $(0, 1, 0)$
- 类别"狗"→ $(0, 0, 1)$

**网络架构**：
- 输入：$d$个特征
- 输出：$q$个类别
- 权重矩阵：$\mathbf{W} \in \mathbb{R}^{d \times q}$
- 偏置：$\mathbf{b} \in \mathbb{R}^{1 \times q}$

### 3.2 Softmax运算

线性层输出 $\mathbf{o}$（未规范化的预测/logit）不能直接作为概率（可能为负，总和不为1）。

**Softmax函数**将logit转换为概率分布：

$$\hat{y}_j = \frac{\exp(o_j)}{\sum_{k=1}^{q} \exp(o_k)}$$

性质：
- 输出非负：$\hat{y}_j \geq 0$
- 总和为1：$\sum_j \hat{y}_j = 1$
- 保持大小顺序：$\arg\max_j \hat{y}_j = \arg\max_j o_j$

**小批量矢量化**：
$$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}, \quad \hat{\mathbf{Y}} = \mathrm{softmax}(\mathbf{O})$$

### 3.3 交叉熵损失（Cross-Entropy Loss）

分类问题的损失函数，基于**最大似然估计**：

$$l(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{j=1}^{q} y_j \log \hat{y}_j$$

由于 $\mathbf{y}$ 是独热编码，只有正确类别的项非零，简化为：

$$l = -\log \hat{y}_{\text{正确类别}}$$

**梯度**：$\partial_{o_j} l = \mathrm{softmax}(\mathbf{o})_j - y_j$（预测概率与真实标签之差）

### 3.4 信息论视角

- **熵（entropy）**：$H[P] = -\sum_j P(j) \log P(j)$，编码所需的信息量
- **交叉熵（cross-entropy）**：$H(P, Q) = -\sum_j P(j) \log Q(j)$，用$Q$编码$P$的代价
- **最小化交叉熵** = 最大化似然 = 最小化传达标签的"惊异"

### 3.5 Softmax回归实现

**从零实现关键代码**：

```python
# Softmax运算
def softmax(O):
    O_exp = torch.exp(O)
    partition = O_exp.sum(dim=1, keepdim=True)
    return O_exp / partition

# 交叉熵损失
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])

# 分类准确率
def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(dim=1)
    return float((y_hat.type(y.dtype) == y).sum())
```

**简洁实现**：

```python
from torch import nn

# 使用内置函数
loss = nn.CrossEntropyLoss()  # 已包含softmax
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
```

## 四、本章核心知识总结

### 线性回归 vs Softmax回归

| 对比项 | 线性回归 | Softmax回归 |
|--------|----------|-------------|
| 任务类型 | 回归（预测数值） | 分类（预测类别） |
| 输出 | 标量 $\hat{y}$ | 概率分布 $\hat{\mathbf{y}}$ |
| 输出层 | 1个神经元 | $q$个神经元 |
| 损失函数 | 均方误差（MSE） | 交叉熵（Cross-Entropy） |
| 激活函数 | 无 | Softmax |

### 关键术语速查表

| 术语 | 英文 | 核心含义 |
|------|------|----------|
| 权重 | weight | 特征对预测的影响 |
| 偏置 | bias | 截距项 |
| 损失函数 | loss function | 衡量预测与真实差距 |
| 均方误差 | mean squared error | 回归损失：$\frac{1}{2}(\hat{y}-y)^2$ |
| 解析解 | analytical solution | 直接公式计算的解 |
| 梯度下降 | gradient descent | 沿梯度方向更新参数 |
| 学习率 | learning rate | 控制更新步长 |
| 批量大小 | batch size | 每次更新的样本数 |
| 超参数 | hyperparameter | 手动设定，不在训练中更新 |
| 独热编码 | one-hot encoding | 类别用向量表示 |
| Softmax | softmax | 将logit转为概率 |
| 交叉熵 | cross-entropy | 分类损失函数 |
| 准确率 | accuracy | 正确预测比例 |
| 全连接层 | fully-connected layer | 每个输入与输出相连 |
| 迭代周期 | epoch | 遍历整个数据集一次 |